# Stage 2 — two-parser CPU collector
No GPU or adapter required. Provide **stage2_dual_source.zip**, **stage2_dual_selective_proposed.zip** and **stage2_dual_tools_proposed.zip**. Inputs must come from the same source pack, match software/Python versions, and cover the same complete 100 tasks. Partial/smoke/scripted or mixed-method/version bundles are refused. Results report correct acceptance, incorrect acceptance, withholding, repairs, parser costs and every parser tool failure.

Kaggle: attach all three ZIPs as datasets and enable Internet. Run all on CPU.


In [ ]:
import datetime,hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR=WORK/('run_'+RUN_ID);RUN_DIR.mkdir()
DRIVE_RESULTS=None
def save_results():
    if DRIVE_RESULTS is not None:
        DRIVE_RESULTS.mkdir(parents=True,exist_ok=True)
        for p in RUN_DIR.iterdir():
            if p.is_file() and p.suffix in ['.zip','.log','.json','.txt']:shutil.copy2(p,DRIVE_RESULTS/p.name)
        print('Drive results:',DRIVE_RESULTS)
print('Run folder:',RUN_DIR)


In [ ]:
MANIFEST_SHA='d9e67c22c047a06aa7996efa2bde5f700c3c8dfdae46b65ad8220fbff609c335'
ROOT=WORK/('stage2_dual_source_'+MANIFEST_SHA[:12])
def unpack(raw,read):
    assert hashlib.sha256(raw).hexdigest()==MANIFEST_SHA,'Wrong source ZIP version'
    manifest=json.loads(raw);assert manifest['experiment']=='stage2-dual-parser-v1'
    for name,sha in manifest['sha256'].items():
        relative=Path(name);assert not relative.is_absolute() and '..' not in relative.parts
        data=read(name);assert hashlib.sha256(data).hexdigest()==sha,f'Changed source/input: {name}'
        path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==data,f'Changed working source: {name}'
        else:path.write_bytes(data)
    sidecar=ROOT/'_stage2_dual_manifest.json'
    if sidecar.exists():assert sidecar.read_bytes()==raw,'Changed source manifest'
    else:sidecar.write_bytes(raw)
    print('Verified fresh-run source/data:',len(manifest['sha256']),'files',MANIFEST_SHA)
found=False
for path in INPUT.rglob('manifest.json'):
    if hashlib.sha256(path.read_bytes()).hexdigest()==MANIFEST_SHA:
        unpack(path.read_bytes(),lambda name:(path.parent/name).read_bytes());found=True;break
if not found:
    for path in INPUT.rglob('stage2_dual_source.zip'):
        with zipfile.ZipFile(path) as z:
            assert len(z.namelist())==len(set(z.namelist()))
            prefix='stage2_dual_source/';name=prefix+'manifest.json'
            if name not in z.namelist():continue
            raw=z.read(name)
            if hashlib.sha256(raw).hexdigest()!=MANIFEST_SHA:continue
            unpack(raw,lambda name:z.read(prefix+name));found=True;break
assert found,'Attach/place stage2_dual_source.zip in the input folder.'
PACKAGES=WORK/'stage2_dual_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')


In [ ]:
paths=[]
for method in ['selective','tools']:
    candidates=list(INPUT.rglob('stage2_dual_'+method+'_proposed.zip'))
    by_hash={hashlib.sha256(p.read_bytes()).hexdigest():p for p in candidates}
    assert len(by_hash)==1,f'Need exactly one distinct full bundle for {method}; found {candidates}'
    paths.append(next(iter(by_hash.values())))
OUT=RUN_DIR/'stage2_dual_comparison'
subprocess.run([sys.executable,'-m','eval.stage2_dual.collect',*map(str,paths),'--out',str(OUT)],cwd=ROOT,env=ENV,check=True)
save_results()
from IPython.display import display,FileLink,Markdown
display(Markdown((OUT/'SUMMARY.md').read_text()));display(FileLink(str(OUT.with_suffix('.zip'))))
print('COMPLETE: two full Stage 2 runs verified; no models loaded.')
